# 02 — Generation E2E Test

Run 10 representative PubMed QA queries through retrieval + reranking + quantized generation.
Manually inspect grounding quality and citation behavior.

In [1]:
import sys
from pathlib import Path

CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path("/teamspace/studios/this_studio"),
]
ROOT = None
for candidate in CANDIDATES:
    if (candidate / "src").exists() and (candidate / "data").exists():
        ROOT = candidate.resolve()
        break

if ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing both src/ and data/.")

for p in [ROOT, ROOT / "src"]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

print('Root:', ROOT)

Root: /teamspace/studios/this_studio


In [2]:
from config.config import SEED
from src.utils.seed import set_seed

set_seed(SEED)
print(f'Seed set to {SEED}')

Seed set to 42


In [4]:
import importlib, gc, os
import config.config as _config_mod
import src.model.loader as _loader_mod
import src.model.llm_wrapper as _llm_mod
import src.model.prompts as _prompts_mod
import src.retrieval.hybrid as _hybrid_mod
import src.retrieval.reranker as _reranker_mod
importlib.reload(_config_mod)
app_config = _config_mod
importlib.reload(_loader_mod)
importlib.reload(_llm_mod)
importlib.reload(_prompts_mod)
importlib.reload(_hybrid_mod)
importlib.reload(_reranker_mod)

# -- Free any VRAM left over from a previous run --
for _var in ("retriever", "reranker", "loaded", "llm"):
    if _var in dir():
        del globals()[_var]
from src.utils.memory import flush_gpu
flush_gpu()

# -- Load the project-configured inference model --
from src.model.llm_wrapper import QuantizedHFLLM, register_llm

loaded = _loader_mod.load_model_and_tokenizer()
llm = QuantizedHFLLM(
    model=loaded.model,
    tokenizer=loaded.tokenizer,
    max_new_tokens=getattr(app_config, "GENERATION_MAX_NEW_TOKENS", 96),
    min_new_tokens=getattr(app_config, "GENERATION_MIN_NEW_TOKENS", 16),
    temperature=getattr(app_config, "GENERATION_TEMPERATURE", 0.0),
    context_window=getattr(app_config, "INFERENCE_CONTEXT_WINDOW", 512),
    do_sample=False,
    repetition_penalty=1.05,
    top_p=1.0,
    top_k=1,
    num_beams=1,
)
register_llm(llm)
flush_gpu()

# -- Retrieval stack follows the project defaults --
embedding_device = getattr(app_config, "EMBEDDING_DEVICE", "auto")
os.environ["SENTENCE_TRANSFORMERS_USE_TORCH_DEVICE"] = "cuda" if embedding_device == "auto" else embedding_device

from src.retrieval.hybrid import load_hybrid_retriever
from src.retrieval.reranker import build_reranker

retriever = load_hybrid_retriever(similarity_top_k=getattr(app_config, "RETRIEVAL_SIMILARITY_TOP_K", 6))
reranker = build_reranker(top_n=getattr(app_config, "RERANK_TOP_N", 2))

print("Retriever:", type(retriever).__name__)
print("Reranker :", type(reranker).__name__)
print("LLM      :", type(llm).__name__)
print("Model id :", getattr(llm.model.config, "_name_or_path", app_config.INFERENCE_MODEL_ID))
print("Model dev:", llm.model.device)
print("HF map   :", getattr(llm.model, "hf_device_map", "<none>"))

Retriever: QueryFusionRetriever
Reranker : SentenceTransformerRerank
LLM      : QuantizedHFLLM
Model id : BioMistral/BioMistral-7B
Model dev: cuda:0
HF map   : {'': 0}


In [6]:
from llama_index.core.schema import QueryBundle
from config import config as app_config
from src.model.prompts import build_generation_prompt
import time

ANSWER_TIMEOUT_SECONDS = getattr(app_config, 'ANSWER_TIMEOUT_SECONDS', 120)

def retrieve_context(query: str):
    qb = QueryBundle(query_str=query)
    candidates = retriever.retrieve(qb)
    reranked = reranker.postprocess_nodes(candidates, query_bundle=qb)
    return reranked

def answer_query(query: str, top_k: int = 2) -> dict:
    start = time.perf_counter()
    nodes = retrieve_context(query)[:top_k]
    context_blocks = []
    for node in nodes:
        chunk_id = getattr(node, 'node_id', None) or node.metadata.get('chunk_id') or 'unknown_chunk'
        context_blocks.append((chunk_id, node.get_content()[:900]))

    prompt = build_generation_prompt(query=query, context_blocks=context_blocks)
    response = llm.complete(prompt, max_time=ANSWER_TIMEOUT_SECONDS)
    elapsed = time.perf_counter() - start

    if elapsed > ANSWER_TIMEOUT_SECONDS:
        raise TimeoutError(
            f'Answer exceeded budget: {elapsed:.1f}s > {ANSWER_TIMEOUT_SECONDS}s'
        )

    return {
        'query': query,
        'answer': response.text,
        'nodes': nodes,
        'elapsed_s': elapsed,
    }

In [7]:
TEST_QUERIES = [
    'Does metformin reduce cardiovascular mortality in type 2 diabetes?',
    'Is cognitive behavioural therapy effective for treatment-resistant depression?',
    'What is the role of BRCA1 mutation in breast cancer risk?',
    'Can aspirin prevent colorectal cancer in high-risk patients?',
    'Does sleep deprivation impair immune function?',
    'Is vitamin D supplementation beneficial for osteoporosis prevention?',
    'Do statins reduce all-cause mortality in primary prevention?',
    'Can probiotics reduce antibiotic-associated diarrhea in adults?',
    'Is intermittent fasting effective for weight loss compared with calorie restriction?',
    'Does omega-3 supplementation improve cognitive outcomes in older adults?'
]

print(f'{len(TEST_QUERIES)} queries loaded')

10 queries loaded


In [8]:
# -- Diagnostic: single query with explicit retrieval/gen timing and full budget --
import time
import torch
from config import config as app_config
from src.model.prompts import build_generation_prompt

ANSWER_TIMEOUT_SECONDS = getattr(app_config, 'ANSWER_TIMEOUT_SECONDS', 120)
GEN_MAX_TOKENS = min(app_config.GENERATION_MAX_NEW_TOKENS, 96)
GEN_MIN_TOKENS = min(app_config.GENERATION_MIN_NEW_TOKENS, 24)

test_q = TEST_QUERIES[0]
t0 = time.perf_counter()

t_retrieval_start = time.perf_counter()
nodes = retrieve_context(test_q)[:2]
retrieval_elapsed = time.perf_counter() - t_retrieval_start

context_blocks = []
for node in nodes:
    chunk_id = getattr(node, "node_id", "unk")
    context_blocks.append((chunk_id, node.get_content()[:700]))

prompt = build_generation_prompt(query=test_q, context_blocks=context_blocks)
print(f"Prompt length (chars): {len(prompt)}")
print(f"Input chunks: {len(context_blocks)}")
print(f"Retrieval time: {retrieval_elapsed:.2f}s")

remaining_budget = max(20.0, ANSWER_TIMEOUT_SECONDS - retrieval_elapsed)
print(f"Generation budget: {remaining_budget:.2f}s")

t_gen_start = time.perf_counter()
response = llm.complete(
    prompt,
    max_new_tokens=GEN_MAX_TOKENS,
    min_new_tokens=GEN_MIN_TOKENS,
    max_time=remaining_budget,
    do_sample=False,
    top_k=1,
    top_p=1.0,
    num_beams=1,
)
gen_elapsed = time.perf_counter() - t_gen_start
elapsed = time.perf_counter() - t0

answer_text = (response.text or "").strip()
print(f"Generation time: {gen_elapsed:.2f}s")
print(f"Total elapsed: {elapsed:.2f}s")
print(
    f"Budget check: {'OK' if elapsed <= ANSWER_TIMEOUT_SECONDS else 'TIMEOUT'} "
    f"(limit={ANSWER_TIMEOUT_SECONDS}s)"
)
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU allocated (GB): {torch.cuda.memory_allocated()/1024**3:.2f}")

if not answer_text or len(answer_text.split()) < 12:
    raise RuntimeError(
        "Output too short/invalid under current budget. Reduce retrieval cost or increase answer budget."
    )

print(f"\nAnswer preview:\n{answer_text[:500]!r}")

Prompt length (chars): 1401
Input chunks: 2
Retrieval time: 0.47s
Generation budget: 119.53s
Generation time: 6.04s
Total elapsed: 6.51s
Budget check: OK (limit=120s)
GPU available: True
GPU allocated (GB): 3.98

Answer preview:
'True. The reasoning is that the study is an open-label, randomized trial.'


In [9]:
import textwrap

generation_results = []
for idx, query in enumerate(TEST_QUERIES, start=1):
    result = answer_query(query)
    generation_results.append(result)

    print('=' * 100)
    print(f'[{idx}] QUERY: {query}')
    print('-' * 100)
    print('ANSWER:')
    print(textwrap.fill(result['answer'], width=100))
    print('\nTOP CONTEXT CHUNKS:')
    for rank, node in enumerate(result['nodes'], start=1):
        chunk_id = getattr(node, 'node_id', None) or node.metadata.get('chunk_id') or 'unknown_chunk'
        preview = node.get_content()[:180].replace('\n', ' ')
        print(f'  [{rank}] {chunk_id}: {preview}...')

print('\nAll generation queries processed.')

[1] QUERY: Does metformin reduce cardiovascular mortality in type 2 diabetes?
----------------------------------------------------------------------------------------------------
ANSWER:
, hypoglycemia, weight gain, and patient satisfaction.'], 'labels': ['TITLE', 'STUDY DESIGN',
'POPULATION', 'INTERVENTIONS', 'CONDITIONS', 'OUTCOMES MEASURED'], 'meshes': ['Analysis of
Variance', 'Diabetes Mellitus, Type 2', 'Drug Therapy, Combination', 'Female',

TOP CONTEXT CHUNKS:
  [1] 15125825_003: Fear of self-injecting and self-testing did not differ.'], 'labels': ['OBJECTIVE', 'STUDY DESIGN', 'POPULATION', 'OUTCOMES MEASURED', 'RESULTS'], 'meshes': ['Analysis of Variance',...
  [2] 15125825_001: {'contexts': ['To evaluate the effects of insulin 30/70 twice daily or bedtime isophane (NPH) insulin plus continued sulfonylurea and metformin in patients with type 2 diabetes in ...
[2] QUERY: Is cognitive behavioural therapy effective for treatment-resistant depression?
------------------------------

In [10]:
import pandas as pd

rows = []
for item in generation_results:
    rows.append({
        'query': item['query'][:80],
        'answer_preview': item['answer'][:220].replace('\n', ' '),
        'num_context_chunks': len(item['nodes']),
    })

df_gen = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 180)
df_gen

,query,answer_preview,num_context_chunks
0,Does metformin reduce cardiovascular mortality in type 2 diabetes?,", hypoglycemia, weight gain, and patient satisfaction.'], 'labels': ['TITLE', 'STUDY DESIGN', 'POPULATION', 'INTERVENTIONS', 'CONDITIONS', 'OUTCOMES MEASURED'], 'meshes': ['Ana...",2
1,Is cognitive behavioural therapy effective for treatment-resistant depression?,"Yes, cognitive behavioral therapy is effective for treatment-resistant depression. [26485091_001] and [7c358a9f-d6c6-41c9-9ebe-37986d5c3765] both support this conclusion.",2
2,What is the role of BRCA1 mutation in breast cancer risk?,"', 'Polymorphism, Genetic', 'Protein Structure Prediction', 'Risk Assessment', 'SNP', 'SNP Analysis', 'SNP Genotyping']}",2
3,Can aspirin prevent colorectal cancer in high-risk patients?,"of Life', 'Risk Assessment', 'Risk Management', 'Risk-Benefit Analysis', 'Risk-to-Benefit Ratio', 'Retrospective Studies', 'Time Series Analysis'], 'reasoning_required_pred': [...",2
4,Does sleep deprivation impair immune function?,"'Female', 'Humans', 'Male', 'Middle Aged', 'Near Infrared Spectroscopy', 'Pediatric', 'Physiologic', 'Sleep', 'Sleep Disturbances', 'Sleep Wake Disorders', 'Time Series Analysi...",2
5,Is vitamin D supplementation beneficial for osteoporosis prevention?,"'s'], 'evidence_required_pred': ['y', 'e', 's'], 'evidence_free_pred': ['y', 'e', 's'], 'evidence_pred': ['evidence_free', 'evidence_free', 'evidence_free', 'evidence_free', 'e...",2
6,Do statins reduce all-cause mortality in primary prevention?,"postoperative AF: those with preoperative statin therapy (Post-statin group, n = 100) and those without it (Post-non-statin group, n = 102).', 'The incidence of postoperative A...",2
7,Can probiotics reduce antibiotic-associated diarrhea in adults?,"e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', '",2
8,Is intermittent fasting effective for weight loss compared with calorie restrict,"Yes, intermittent fasting is effective for weight loss compared with calorie restriction.",2
9,Does omega-3 supplementation improve cognitive outcomes in older adults?,"The age-related WM deficit was not greatly affected, however, and the training gains did not transfer to the other cognitive tasks. In fact, participants attempted to adapt the...",2


In [ ]:
import torch

if torch.cuda.is_available():
    print(torch.cuda.memory_summary())
else:
    print('CUDA not available; VRAM summary skipped.')

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4074 MiB |   6782 MiB |   1915 GiB |   1911 GiB |
|       from large pool |   3938 MiB |   6782 MiB |   1766 GiB |   1762 GiB |
|       from small pool |    136 MiB |    203 MiB |    148 GiB |    148 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4074 MiB |   6782 MiB |   1915 GiB |   1911 GiB |
|       from large pool |   3938 MiB |   6782 MiB |   1766 GiB |

: 

## Manual Quality Checklist

- Answers cite chunk IDs inline using `[chunk_id]` style.
- Claims are supported by retrieved context snippets.
- Unsupported questions explicitly state insufficient context.
- Response quality remains stable across all 10 test queries.
- CUDA memory summary indicates adequate headroom for generation.